In [3]:
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless
!wget -q https://downloads.apache.org/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q pyspark findspark


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  openjdk-11-jre-headless
Suggested packages:
  openjdk-11-demo openjdk-11-source libnss-mdns fonts-dejavu-extra
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  openjdk-11-jdk-headless openjdk-11-jre-headless
0 upgraded, 2 newly installed, 0 to remove and 55 not upgraded.
Need to get 116 MB of archives.
After this operation, 258 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 openjdk-11-jre-headless amd64 11.0.30+7-1ubuntu1~22.04 [42.6 MB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64

In [4]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"


In [5]:
import findspark
findspark.init("/content/spark-3.5.0-bin-hadoop3")

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("BDA_Assignment_2") \
    .getOrCreate()

spark


In [6]:
!pip install datasets


In [8]:
from datasets import load_dataset
from pyspark.sql.functions import col

languages = ["hin_Deva", "guj_Gujr", "tam_Taml"]  # 3 languages
MAX_ROWS_PER_LANG = 10000

rows = []

for lang in languages:
    ds = load_dataset(
        "ai4bharat/IndicCorpV2",
        split=lang,
        streaming=True
    )

    for i, row in enumerate(ds):
        if i >= MAX_ROWS_PER_LANG:
            break
        rows.append((row["text"], lang))


In [9]:
df = spark.createDataFrame(rows, ["text", "language"])
df.show(5)
df.groupBy("language").count().show()


+--------------------+--------+
|                text|language|
+--------------------+--------+
|लोगों को बिलों सं...|hin_Deva|
|                    |hin_Deva|
|इनेलो 1987 में उस...|hin_Deva|
|                    |hin_Deva|
|जहां आई थी तबाही ...|hin_Deva|
+--------------------+--------+
only showing top 5 rows
+--------+-----+
|language|count|
+--------+-----+
|guj_Gujr|10000|
|hin_Deva|10000|
|tam_Taml|10000|
+--------+-----+



In [10]:
from pyspark.sql.functions import length, trim

total_lines = df.count()
empty_lines = df.filter(
    col("text").isNull() | (length(trim(col("text"))) == 0)
).count()

print("Total lines:", total_lines)
print("Empty lines:", empty_lines)


Total lines: 30000
Empty lines: 15000


In [11]:
from pyspark.sql.functions import min, max, avg

df_len = df.withColumn("sentence_length", length(col("text")))

df_len.select(
    min("sentence_length").alias("Min"),
    max("sentence_length").alias("Max"),
    avg("sentence_length").alias("Avg")
).show()


+---+-----+------------------+
|Min|  Max|               Avg|
+---+-----+------------------+
|  0|41210|140.98266666666666|
+---+-----+------------------+



In [12]:
# Check schema and a few rows to confirm data loaded correctly
df.printSchema()
df.show(5, truncate=80)


root
 |-- text: string (nullable = true)
 |-- language: string (nullable = true)

+--------------------------------------------------------------------------------+--------+
|                                                                            text|language|
+--------------------------------------------------------------------------------+--------+
|                                   लोगों को बिलों संबंधी सुविधा देना ही उनका काम|hin_Deva|
|                                                                                |hin_Deva|
|इनेलो 1987 में उस वक्त ऐसे ही दोराहे पर खड़ी थी, जब पूर्व उपप्रधानमंत्री देवी...|hin_Deva|
|                                                                                |hin_Deva|
|                                जहां आई थी तबाही उस घाटी क्षेत्र में खतरा ज्यादा|hin_Deva|
+--------------------------------------------------------------------------------+--------+
only showing top 5 rows


In [13]:
from pyspark.sql.functions import col, length, trim

# Total number of rows (sentences)
total_lines = df.count()

# Empty or malformed lines:
# - text is NULL
# - OR text becomes empty after trimming spaces
empty_lines = df.filter(
    col("text").isNull() | (length(trim(col("text"))) == 0)
).count()

print("Total number of lines:", total_lines)
print("Number of empty/malformed lines:", empty_lines)


Total number of lines: 30000
Number of empty/malformed lines: 15000


In [14]:
from pyspark.sql.functions import min, max, avg

# Remove empty sentences
clean_df = df.filter(length(trim(col("text"))) > 0)

# Add a new column: sentence length (number of characters)
clean_df = clean_df.withColumn("sentence_length", length(col("text")))

# Compute min, max, and average sentence length
clean_df.select(
    min("sentence_length").alias("Min_Length"),
    max("sentence_length").alias("Max_Length"),
    avg("sentence_length").alias("Avg_Length")
).show()


+----------+----------+-----------------+
|Min_Length|Max_Length|       Avg_Length|
+----------+----------+-----------------+
|         5|     41210|281.9653333333333|
+----------+----------+-----------------+



In [15]:
from pyspark.sql.functions import explode, split, lower

# Convert text to lowercase and split into words
token_df = clean_df.withColumn(
    "token",
    explode(split(lower(col("text")), " "))
)

# Remove very short or empty tokens
token_df = token_df.filter(length(col("token")) > 2)

# Select 3 languages for analysis
languages = ["hin_Deva", "guj_Gujr", "tam_Taml"]

for lang in languages:
    print(f"\nTop 10 most common tokens for {lang}:")

    token_df.filter(col("language") == lang) \
        .groupBy("token") \
        .count() \
        .orderBy(col("count").desc()) \
        .show(10, truncate=False)



Top 10 most common tokens for hin_Deva:
+-----+-----+
|token|count|
+-----+-----+
|में  |9066 |
|है।  |3697 |
|लिए  |1922 |
|नहीं |1525 |
|हैं। |1403 |
|किया |1196 |
|करने |1057 |
|कहा  |931  |
|साथ  |926  |
|बाद  |912  |
+-----+-----+
only showing top 10 rows

Top 10 most common tokens for guj_Gujr:
+-------+-----+
|token  |count|
+-------+-----+
|છે.    |5490 |
|અને    |3955 |
|માટે   |1719 |
|કરી    |1300 |
|સાથે   |1096 |
|છે,    |816  |
|હતી.   |804  |
|આવી    |739  |
|કરવામાં|693  |
|દ્વારા |628  |
+-------+-----+
only showing top 10 rows

Top 10 most common tokens for tam_Taml:
+-------+-----+
|token  |count|
+-------+-----+
|ஒரு    |1313 |
|மற்றும்|1030 |
|இந்த   |911  |
|என்று  |903  |
|என்ற   |457  |
|அந்த   |444  |
|அவர்   |396  |
|இது    |391  |
|உள்ள   |370  |
|மக்கள் |301  |
+-------+-----+
only showing top 10 rows


In [16]:
from pyspark.ml.feature import Tokenizer, HashingTF, IDF


In [17]:
# Step 1: Tokenize text into words
tokenizer = Tokenizer(inputCol="text", outputCol="words")
words_df = tokenizer.transform(clean_df)


In [18]:
# Step 2: Convert words into term frequency vectors
hashingTF = HashingTF(
    inputCol="words",
    outputCol="rawFeatures",
    numFeatures=10000
)

tf_df = hashingTF.transform(words_df)


In [19]:
# Step 3: Compute IDF (inverse document frequency)
idf = IDF(inputCol="rawFeatures", outputCol="tfidf")
idf_model = idf.fit(tf_df)

tfidf_df = idf_model.transform(tf_df)

# Show TF-IDF vectors
tfidf_df.select("language", "tfidf").show(5, truncate=False)


+--------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------